# DPS Departures — Preprocessing, Agregasi Harian & Merge BMKG

**Input:**
- `dps_departures_YYYY-MM.csv` — data penerbangan mentah per bulan (Mar 2025 – Mei 2026)
- `*.xlsx` — laporan iklim harian BMKG Stasiun Ngurah Rai

**Output:** `dps_bmkg_merged.csv` — 1 baris per hari, 21 kolom penerbangan + cuaca

**Aturan kategorisasi penerbangan (IATA standard):**
- `early`     : berangkat > 1 menit lebih awal
- `ontime`    : delay antara −1 s/d +15 menit
- `delay`     : delay > 15 menit
- `cancelled` : status Cancelled atau Diverted

## 0. Install & Import

In [7]:
!pip install openpyxl --quiet

import pandas as pd
import numpy as np
import glob, os, re
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Siap!')

Siap!


---
# BAGIAN 1 — PREPROCESSING DATA PENERBANGAN

## 1. Load Semua File CSV Penerbangan

In [8]:
DATA_DIR           = Path('.')   # sesuaikan jika file ada di folder lain
ONTIME_THRESHOLD_MIN = 15

months     = pd.period_range('2025-03', '2026-05', freq='M')
file_names = [f'dps_departures_{m}.csv' for m in months]

print(f'Mencari {len(file_names)} file:')
for f in file_names:
    path   = DATA_DIR / f
    status = 'Ada        ' if path.exists() else 'Tidak Ada  '
    print(f'  {status} {f}')

Mencari 15 file:
  Ada         dps_departures_2025-03.csv
  Ada         dps_departures_2025-04.csv
  Ada         dps_departures_2025-05.csv
  Ada         dps_departures_2025-06.csv
  Ada         dps_departures_2025-07.csv
  Ada         dps_departures_2025-08.csv
  Ada         dps_departures_2025-09.csv
  Ada         dps_departures_2025-10.csv
  Ada         dps_departures_2025-11.csv
  Ada         dps_departures_2025-12.csv
  Ada         dps_departures_2026-01.csv
  Ada         dps_departures_2026-02.csv
  Ada         dps_departures_2026-03.csv
  Ada         dps_departures_2026-04.csv
  Ada         dps_departures_2026-05.csv


In [9]:
dfs, missing = [], []

for f in file_names:
    path = DATA_DIR / f
    if path.exists():
        df_tmp = pd.read_csv(path, dtype={'tanggal': str})
        df_tmp['_source_file'] = f
        dfs.append(df_tmp)
    else:
        missing.append(f)

if missing:
    print(f'{len(missing)} file tidak ditemukan, dilewati: {missing}')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'Total baris gabungan: {len(df_raw):,}')
print(f'Rentang tanggal     : {df_raw["tanggal"].min()} s/d {df_raw["tanggal"].max()}')
print(f'\nDistribusi status:')
print(df_raw['status'].value_counts())

Total baris gabungan: 92,617
Rentang tanggal     : 2025-03-01 s/d 2026-05-31

Distribusi status:
status
Landed       87719
Cancelled     1442
Diverted        38
Name: count, dtype: int64


## 2. Parse Kolom `delay_berangkat` → Menit (Numerik)

In [10]:
def parse_delay(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ('on time', 'ontime', '0', ''):
        return 0.0
    m = re.match(r'(\d+)\s*min\s*(late|early)', val)
    if m:
        menit = int(m.group(1))
        return menit if m.group(2) == 'late' else -menit
    try:
        return float(val)
    except ValueError:
        return np.nan

df_raw['delay_menit'] = df_raw['delay_berangkat'].apply(parse_delay)

n_unparsed   = df_raw['delay_menit'].isna().sum()
n_cancelled  = df_raw['status'].isin(['Cancelled', 'Diverted']).sum()
n_null_operated = df_raw[
    df_raw['delay_menit'].isna() & ~df_raw['status'].isin(['Cancelled', 'Diverted'])
].shape[0]

print(f'Total null delay_menit          : {n_unparsed:,}')
print(f'  → dari Cancelled/Diverted     : {n_cancelled:,}  (wajar)')
print(f'  → dari Landed/operated        : {n_null_operated:,}  (akan di-impute)')

Total null delay_menit          : 5,631
  → dari Cancelled/Diverted     : 1,480  (wajar)
  → dari Landed/operated        : 4,151  (akan di-impute)


## 3. Impute Null pada Operated Flights

Flight `Landed` dengan `delay_berangkat` kosong & `jadwal_lokal == berangkat_aktual` → impute **0 menit**.

In [11]:
mask_operated_null = (
    df_raw['delay_menit'].isna() &
    ~df_raw['status'].isin(['Cancelled', 'Diverted'])
)

operated_null = df_raw[mask_operated_null][[
    'tanggal', 'flight_iata', 'airline',
    'jadwal_lokal', 'berangkat_aktual', 'delay_berangkat', 'status'
]]
print(f'Operated flights dengan null delay: {len(operated_null)}')
display(operated_null.head(10))

same_time = (
    df_raw.loc[mask_operated_null, 'jadwal_lokal'] ==
    df_raw.loc[mask_operated_null, 'berangkat_aktual']
).sum()
print(f'\nDari {mask_operated_null.sum()} baris null: {same_time} punya jadwal == berangkat_aktual → impute 0')

df_raw.loc[mask_operated_null, 'delay_menit'] = 0.0
print('Impute selesai.')

Operated flights dengan null delay: 4151


,tanggal,flight_iata,airline,jadwal_lokal,berangkat_aktual,delay_berangkat,status
22,2025-03-01,JT856,Lion Air,07:00,NaN,NaN,NaN
67,2025-03-01,0B772,Blue Air,12:50,NaN,NaN,NaN
127,2025-03-01,IN281,NAM Air,17:45,NaN,NaN,NaN
147,2025-03-01,IU727,SW Italia,20:55,NaN,NaN,NaN
151,2025-03-01,IP105,Pelita Air Service,21:15,NaN,NaN,NaN
185,2025-03-02,JT856,Lion Air,07:00,NaN,NaN,NaN
220,2025-03-03,SJ726,Sriwijaya Air,01:00,NaN,NaN,NaN
236,2025-03-03,8B5101,TransNusa,10:30,NaN,NaN,NaN
259,2025-03-03,0B772,Blue Air,12:50,NaN,NaN,NaN
287,2025-03-03,8B5103,TransNusa,15:00,NaN,NaN,NaN



Dari 4151 baris null: 717 punya jadwal == berangkat_aktual → impute 0
Impute selesai.


## 4. Kategorisasi Per-Flight

In [12]:
def kategorisasi(row):
    if row['status'] in ('Cancelled', 'Diverted'):
        return 'cancelled'
    d = row['delay_menit']
    if pd.isna(d):
        return 'unknown'
    if d < -1:
        return 'early'
    elif d <= ONTIME_THRESHOLD_MIN:
        return 'ontime'
    else:
        return 'delay'

df_raw['kategori'] = df_raw.apply(kategorisasi, axis=1)

print('Distribusi kategori (semua bulan):')
print(df_raw['kategori'].value_counts())
print(f'\nTotal unknown (harusnya 0): {(df_raw["kategori"] == "unknown").sum()}')

Distribusi kategori (semua bulan):
kategori
delay        43144
ontime       40368
early         7625
cancelled     1480
Name: count, dtype: int64

Total unknown (harusnya 0): 0


## 5. Agregasi Per Hari

In [13]:
df_raw['tanggal'] = pd.to_datetime(df_raw['tanggal'])

agg_counts = (
    df_raw.groupby(['tanggal', 'kategori'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['early', 'ontime', 'delay', 'cancelled', 'unknown']:
    if col not in agg_counts.columns:
        agg_counts[col] = 0

avg_delay = (
    df_raw[df_raw['kategori'] == 'delay']
    .groupby('tanggal')['delay_menit']
    .mean()
    .reset_index()
    .rename(columns={'delay_menit': 'avg_delay_menit'})
)

daily = agg_counts.merge(avg_delay, on='tanggal', how='left')

daily['total_flight'] = daily['early'] + daily['ontime'] + daily['delay'] + daily['cancelled'] + daily['unknown']
daily['operated']     = daily['early'] + daily['ontime'] + daily['delay'] + daily['unknown']
daily['pct_ontime']   = np.where(
    daily['operated'] > 0,
    (daily['early'] + daily['ontime']) / daily['operated'] * 100,
    np.nan
)
daily['pct_delay'] = np.where(
    daily['operated'] > 0,
    daily['delay'] / daily['operated'] * 100,
    np.nan
)
daily['avg_delay_menit'] = daily['avg_delay_menit'].round(1)
daily['pct_ontime']      = daily['pct_ontime'].round(1)
daily['pct_delay']       = daily['pct_delay'].round(1)
daily = daily.drop(columns=['unknown'], errors='ignore')
daily = daily[[
    'tanggal', 'total_flight', 'early', 'ontime', 'delay',
    'cancelled', 'operated', 'pct_ontime', 'pct_delay', 'avg_delay_menit'
]].sort_values('tanggal').reset_index(drop=True)

print(f'Jumlah baris harian : {len(daily)}')
print(f'Rentang             : {daily["tanggal"].min().date()} s/d {daily["tanggal"].max().date()}')
display(daily.head(10))

Jumlah baris harian : 457
Rentang             : 2025-03-01 s/d 2026-05-31


,tanggal,total_flight,early,ontime,delay,cancelled,operated,pct_ontime,pct_delay,avg_delay_menit
0,2025-03-01,170,36,80,44,10,160,72.50,27.50,27.90
1,2025-03-02,43,13,24,5,1,42,88.10,11.90,28.60
2,2025-03-03,150,35,74,37,4,146,74.70,25.30,31.50
3,2025-03-04,160,39,90,28,3,157,82.20,17.80,26.40
4,2025-03-05,171,28,109,33,1,170,80.60,19.40,26.70
5,2025-03-06,173,46,82,27,18,155,82.60,17.40,30.20
6,2025-03-07,186,42,98,31,15,171,81.90,18.10,33.70
7,2025-03-08,173,17,63,79,14,159,50.30,49.70,37.50
8,2025-03-09,189,34,91,53,11,178,70.20,29.80,29.50
9,2025-03-10,176,27,95,46,8,168,72.60,27.40,31.00


## 6. Validasi Agregasi

In [14]:
print('=' * 55)
print('VALIDASI')
print('=' * 55)

total_daily = daily['total_flight'].sum()
total_raw   = len(df_raw)
print(f'Total flight (raw)   : {total_raw:,}')
print(f'Total flight (daily) : {total_daily:,}')
print(f'Match                : {"✅ aman" if total_daily == total_raw else "❌ mismatch"}')

print()
print('Null per kolom:')
print(daily.isna().sum())

print()
print('Statistik ringkas:')
display(daily[['total_flight','early','ontime','delay','cancelled','pct_ontime','pct_delay','avg_delay_menit']].describe().round(2))

VALIDASI
Total flight (raw)   : 92,617
Total flight (daily) : 92,617
Match                : ✅ aman

Null per kolom:
tanggal            0
total_flight       0
early              0
ontime             0
delay              0
cancelled          0
operated           0
pct_ontime         0
pct_delay          0
avg_delay_menit    0
dtype: int64

Statistik ringkas:


,total_flight,early,ontime,delay,cancelled,pct_ontime,pct_delay,avg_delay_menit
count,457.00,457.00,457.00,457.00,457.00,457.00,457.00,457.00
mean,202.66,16.68,88.33,94.41,3.24,53.09,46.91,30.98
std,23.45,7.40,14.74,22.95,4.61,8.62,8.62,1.96
min,43.00,3.00,24.00,5.00,0.00,32.70,11.90,23.90
25%,196.00,11.00,79.00,81.00,0.00,47.10,41.70,29.70
50%,204.00,16.00,88.00,96.00,1.00,52.50,47.50,31.00
75%,210.00,21.00,96.00,109.00,4.00,58.30,52.90,32.40
max,517.00,46.00,221.00,259.00,28.00,88.10,67.30,37.50


In [15]:
print('10 hari dengan on-time rate terendah:')
display(
    daily.nsmallest(10, 'pct_ontime')[[
        'tanggal', 'total_flight', 'early', 'ontime',
        'delay', 'cancelled', 'pct_ontime', 'avg_delay_menit'
    ]]
)

10 hari dengan on-time rate terendah:


,tanggal,total_flight,early,ontime,delay,cancelled,pct_ontime,avg_delay_menit
116,2025-06-25,209,13,55,140,1,32.70,36.20
111,2025-06-20,209,11,58,137,3,33.50,34.80
119,2025-06-28,205,10,60,135,0,34.10,34.10
146,2025-07-25,215,6,67,140,2,34.30,32.30
110,2025-06-19,205,8,64,129,4,35.80,32.50
125,2025-07-04,219,8,71,140,0,36.10,31.80
382,2026-03-18,215,7,71,136,1,36.40,29.60
114,2025-06-23,204,15,59,128,2,36.60,34.00
117,2025-06-26,207,10,66,130,1,36.90,34.80
294,2025-12-20,207,9,68,130,0,37.20,33.20


---
# BAGIAN 2 — LOAD & PREPROCESSING DATA BMKG

## 7. Load Semua File BMKG

Struktur file BMKG Ngurah Rai:
- Baris 0–7 : metadata stasiun
- Baris 8–38 : data harian (11 kolom)

In [16]:
BMKG_DIR  = '.'   # sesuaikan jika file ada di folder lain
DATA_COLS = ['TANGGAL','TN','TX','TAVG','RH_AVG','RR','SS','FF_X','DDD_X','FF_AVG','DDD_CAR']

bmkg_files = sorted(glob.glob(os.path.join(BMKG_DIR, '*.xlsx')))
records    = []
skipped    = []

for filepath in bmkg_files:
    filename = os.path.basename(filepath)
    raw = pd.read_excel(filepath, sheet_name='Worksheet', header=None)

    if raw.shape[1] < 11:
        skipped.append(filename)
        print(f'  [SKIP] {filename} — kolom kurang ({raw.shape[1]})')
        continue

    data_rows = raw.iloc[8:39, :11].copy()
    data_rows.columns = DATA_COLS
    data_rows = data_rows.dropna(subset=['TANGGAL'])
    data_rows = data_rows[
        data_rows['TANGGAL'].astype(str).str.match(r'\d{2}-\d{2}-\d{4}', na=False)
    ]
    data_rows['SUMBER_FILE'] = filename

    if len(data_rows) > 0:
        records.append(data_rows)
        tgl = pd.to_datetime(data_rows['TANGGAL'], dayfirst=True, errors='coerce')
        print(f'  {filename}: {tgl.min().date()} – {tgl.max().date()} ({len(data_rows)} hari)')
    else:
        skipped.append(filename)
        print(f'  [EMPTY] {filename}')

df_bmkg_raw = pd.concat(records, ignore_index=True)
df_bmkg_raw['TANGGAL'] = pd.to_datetime(df_bmkg_raw['TANGGAL'], dayfirst=True, errors='coerce')
df_bmkg_raw = df_bmkg_raw.dropna(subset=['TANGGAL'])
df_bmkg_raw = df_bmkg_raw.drop_duplicates(subset='TANGGAL', keep='first')
df_bmkg_raw = df_bmkg_raw.sort_values('TANGGAL').reset_index(drop=True)

print(f'\nTotal BMKG : {len(df_bmkg_raw):,} hari')
print(f'Rentang    : {df_bmkg_raw["TANGGAL"].min().date()} s/d {df_bmkg_raw["TANGGAL"].max().date()}')
if skipped:
    print(f'File skip  : {skipped}')

  apr 2025.xlsx: 2025-04-01 – 2025-04-30 (30 hari)
  aug 2025.xlsx: 2025-08-01 – 2025-08-31 (31 hari)
  des 2025.xlsx: 2025-12-01 – 2025-12-31 (31 hari)
  feb 2025.xlsx: 2025-02-01 – 2025-02-28 (28 hari)
  jan 2025.xlsx: 2025-01-01 – 2025-01-31 (31 hari)
  jul 2025.xlsx: 2025-07-01 – 2025-07-31 (31 hari)
  jun 2025.xlsx: 2025-06-01 – 2025-06-30 (30 hari)
  laporan_iklim_harian-april-2026.xlsx: 2026-04-01 – 2026-04-30 (30 hari)
  laporan_iklim_harian-februari-2026.xlsx: 2026-02-01 – 2026-02-28 (28 hari)
  laporan_iklim_harian-januari-2026.xlsx: 2026-01-01 – 2026-01-31 (31 hari)
  laporan_iklim_harian-maret-2026.xlsx: 2026-03-01 – 2026-03-31 (31 hari)
  laporan_iklim_harian-mei-2026.xlsx: 2026-05-01 – 2026-05-29 (29 hari)
  mar 2025.xlsx: 2025-03-01 – 2025-03-31 (31 hari)
  mei 2025.xlsx: 2025-05-01 – 2025-05-31 (31 hari)
  nov 2025.xlsx: 2025-11-01 – 2025-11-30 (30 hari)
  okt 2025.xlsx: 2025-10-01 – 2025-10-31 (31 hari)
  sept 2025.xlsx: 2025-09-01 – 2025-09-30 (30 hari)

Total BMKG : 

## 8. Cleaning BMKG

- Konversi numerik
- Ganti kode error BMKG (8888 / 9999) → NaN
- Validasi range logis

In [17]:
df_bmkg  = df_bmkg_raw.copy()
num_cols = ['TN','TX','TAVG','RH_AVG','RR','SS','FF_X','DDD_X','FF_AVG']

for col in num_cols:
    df_bmkg[col] = pd.to_numeric(df_bmkg[col], errors='coerce')

total_8888, total_9999 = 0, 0
for col in num_cols:
    m8 = df_bmkg[col] == 8888
    m9 = df_bmkg[col] == 9999
    total_8888 += m8.sum()
    total_9999 += m9.sum()
    df_bmkg.loc[m8, col] = np.nan
    df_bmkg.loc[m9, col] = np.nan

# Range logis
df_bmkg.loc[df_bmkg['TN']     <  0,  'TN']     = np.nan
df_bmkg.loc[df_bmkg['TX']     > 45,  'TX']     = np.nan
df_bmkg.loc[df_bmkg['RR']     <  0,  'RR']     = np.nan
df_bmkg.loc[df_bmkg['RH_AVG'] > 100, 'RH_AVG'] = np.nan

df_bmkg['DDD_CAR'] = df_bmkg['DDD_CAR'].astype(str).str.strip().str.upper()
df_bmkg['BULAN']   = df_bmkg['TANGGAL'].dt.month

print(f'Kode 8888 → NaN : {total_8888} nilai')
print(f'Kode 9999 → NaN : {total_9999} nilai')
print(f'\nMissing per kolom sebelum imputasi:')
print(df_bmkg[num_cols].isna().sum().to_string())

Kode 8888 → NaN : 31 nilai
Kode 9999 → NaN : 0 nilai

Missing per kolom sebelum imputasi:
TN         1
TX         0
TAVG       1
RH_AVG     1
RR        31
SS         0
FF_X       0
DDD_X     11
FF_AVG     0


## 9. Imputasi Missing BMKG

Strategi 3-lapis: **Forward fill → Backward fill → Median per bulan**

In [18]:
df_bmkg = df_bmkg.sort_values('TANGGAL').reset_index(drop=True)

for col in num_cols:
    filled      = df_bmkg[col].ffill().bfill()
    median_fill = df_bmkg.groupby('BULAN')[col].transform('median')
    df_bmkg[col + '_FILLED'] = filled.fillna(median_fill)

df_bmkg['RR_IMPUTED'] = df_bmkg['RR'].isna() & df_bmkg['RR_FILLED'].notna()

bins   = [-0.1, 0, 5, 20, 50, 100, 150, 9999]
labels = ['Tidak Hujan','Sangat Ringan','Ringan','Sedang','Lebat','Sangat Lebat','Ekstrem']
df_bmkg['KATEGORI_HUJAN'] = pd.cut(df_bmkg['RR_FILLED'], bins=bins, labels=labels)

filled_cols = [c + '_FILLED' for c in num_cols]
print('Missing setelah imputasi:')
print(df_bmkg[filled_cols].isna().sum().to_string())
print(f'\nBaris RR diimputasi : {df_bmkg["RR_IMPUTED"].sum()}')

Missing setelah imputasi:
TN_FILLED        0
TX_FILLED        0
TAVG_FILLED      0
RH_AVG_FILLED    0
RR_FILLED        0
SS_FILLED        0
FF_X_FILLED      0
DDD_X_FILLED     0
FF_AVG_FILLED    0

Baris RR diimputasi : 31


---
# BAGIAN 3 — MERGE PENERBANGAN + BMKG

## 10. Filter BMKG & Cek Overlap Tanggal

In [19]:
flight_start = daily['tanggal'].min()
flight_end   = daily['tanggal'].max()

df_bmkg_filt = df_bmkg[
    (df_bmkg['TANGGAL'] >= flight_start) &
    (df_bmkg['TANGGAL'] <= flight_end)
].copy()

# Handle hari terakhir yang tidak ada di BMKG (ffill dari hari terdekat)
flight_dates = set(daily['tanggal'].dt.date)
bmkg_dates   = set(df_bmkg_filt['TANGGAL'].dt.date)
only_flight  = sorted(flight_dates - bmkg_dates)

if only_flight:
    print(f'Tanggal penerbangan tanpa BMKG: {only_flight}')
    print('  → mengisi dengan ffill dari hari terakhir BMKG yang ada...')
    last_row = df_bmkg_filt.sort_values('TANGGAL').iloc[-1].copy()
    extra_rows = []
    for d in only_flight:
        row = last_row.copy()
        row['TANGGAL']    = pd.Timestamp(d)
        row['RR_IMPUTED'] = True
        extra_rows.append(row)
    df_bmkg_filt = pd.concat(
        [df_bmkg_filt, pd.DataFrame(extra_rows)], ignore_index=True
    ).sort_values('TANGGAL').reset_index(drop=True)
    print(f'  → {len(extra_rows)} baris ditambahkan (flag rr_imputed=True)')

# Pilih kolom
bmkg_keep = [
    'TANGGAL',
    'TN_FILLED','TX_FILLED','TAVG_FILLED','RH_AVG_FILLED',
    'RR_FILLED','RR_IMPUTED','KATEGORI_HUJAN',
    'SS_FILLED','FF_X_FILLED','FF_AVG_FILLED','DDD_CAR'
]
df_bmkg_filt = df_bmkg_filt[bmkg_keep].rename(columns={'TANGGAL': 'tanggal'})

# Laporan akhir overlap
flight_dates2 = set(daily['tanggal'].dt.date)
bmkg_dates2   = set(df_bmkg_filt['tanggal'].dt.date)
print(f'\nHari penerbangan          : {len(flight_dates2)}')
print(f'Hari BMKG (dalam rentang) : {len(bmkg_dates2)}')
print(f'Overlap                   : {len(flight_dates2 & bmkg_dates2)}')
print(f'Tanpa BMKG                : {len(flight_dates2 - bmkg_dates2)}')

Tanggal penerbangan tanpa BMKG: [datetime.date(2026, 5, 30), datetime.date(2026, 5, 31)]
  → mengisi dengan ffill dari hari terakhir BMKG yang ada...
  → 2 baris ditambahkan (flag rr_imputed=True)

Hari penerbangan          : 457
Hari BMKG (dalam rentang) : 457
Overlap                   : 457
Tanpa BMKG                : 0


## 11. Merge (Left Join)

In [20]:
df_merged = daily.merge(
    df_bmkg_filt,
    on='tanggal',
    how='left',
    validate='1:1'
)

df_merged = df_merged.rename(columns={
    'TN_FILLED'    : 'suhu_min',
    'TX_FILLED'    : 'suhu_max',
    'TAVG_FILLED'  : 'suhu_avg',
    'RH_AVG_FILLED': 'kelembapan',
    'RR_FILLED'    : 'curah_hujan',
    'RR_IMPUTED'   : 'rr_imputed',
    'KATEGORI_HUJAN':'kategori_hujan',
    'SS_FILLED'    : 'penyinaran_jam',
    'FF_X_FILLED'  : 'angin_max_ms',
    'FF_AVG_FILLED': 'angin_avg_ms',
    'DDD_CAR'      : 'arah_angin',
})

print(f'Hasil merge : {len(df_merged):,} baris × {len(df_merged.columns)} kolom')
print(f'Rentang     : {df_merged["tanggal"].min().date()} s/d {df_merged["tanggal"].max().date()}')

cuaca_cols = ['suhu_min','suhu_max','suhu_avg','kelembapan',
              'curah_hujan','penyinaran_jam','angin_max_ms','angin_avg_ms']
mv = df_merged[cuaca_cols].isna().sum()
print(f'\nMissing kolom cuaca:')
print(mv.to_string())

Hasil merge : 457 baris × 21 kolom
Rentang     : 2025-03-01 s/d 2026-05-31

Missing kolom cuaca:
suhu_min          0
suhu_max          0
suhu_avg          0
kelembapan        0
curah_hujan       0
penyinaran_jam    0
angin_max_ms      0
angin_avg_ms      0


## 12. Preview & Statistik Ringkas

In [21]:
display(df_merged.head(10))

,tanggal,total_flight,early,ontime,delay,cancelled,operated,pct_ontime,pct_delay,avg_delay_menit,suhu_min,suhu_max,suhu_avg,kelembapan,curah_hujan,rr_imputed,kategori_hujan,penyinaran_jam,angin_max_ms,angin_avg_ms,arah_angin
0,2025-03-01,170,36,80,44,10,160,72.50,27.50,27.90,25.20,32.30,28.00,81.00,11.40,False,Ringan,8.00,4.00,2.00,E
1,2025-03-02,43,13,24,5,1,42,88.10,11.90,28.60,25.30,32.10,28.70,81.00,1.20,False,Sangat Ringan,8.00,4.00,2.00,E
2,2025-03-03,150,35,74,37,4,146,74.70,25.30,31.50,25.80,32.20,28.60,77.00,0.00,False,Tidak Hujan,8.00,4.00,2.00,E
3,2025-03-04,160,39,90,28,3,157,82.20,17.80,26.40,25.80,31.30,28.50,79.00,1.70,False,Sangat Ringan,8.00,4.00,2.00,E
4,2025-03-05,171,28,109,33,1,170,80.60,19.40,26.70,26.00,32.00,28.80,81.00,0.00,False,Tidak Hujan,8.00,5.00,3.00,E
5,2025-03-06,173,46,82,27,18,155,82.60,17.40,30.20,26.50,32.00,29.00,79.00,0.00,False,Tidak Hujan,8.00,5.00,3.00,E
6,2025-03-07,186,42,98,31,15,171,81.90,18.10,33.70,26.20,32.80,28.70,76.00,0.00,False,Tidak Hujan,8.00,4.00,2.00,E
7,2025-03-08,173,17,63,79,14,159,50.30,49.70,37.50,26.20,31.90,28.40,77.00,0.00,False,Tidak Hujan,8.00,4.00,2.00,E
8,2025-03-09,189,34,91,53,11,178,70.20,29.80,29.50,26.40,31.60,28.50,77.00,0.00,False,Tidak Hujan,7.40,6.00,2.00,E
9,2025-03-10,176,27,95,46,8,168,72.60,27.40,31.00,25.20,32.50,28.00,77.00,3.20,False,Sangat Ringan,2.70,5.00,3.00,W


In [22]:
cols_stat = ['total_flight','pct_ontime','pct_delay','avg_delay_menit',
             'suhu_avg','curah_hujan','angin_max_ms','kelembapan']
display(df_merged[cols_stat].describe().round(2))

,total_flight,pct_ontime,pct_delay,avg_delay_menit,suhu_avg,curah_hujan,angin_max_ms,kelembapan
count,457.00,457.00,457.00,457.00,457.00,457.00,457.00,457.00
mean,202.66,53.09,46.91,30.98,27.63,8.30,5.85,80.33
std,23.45,8.62,8.62,1.96,0.95,19.89,1.89,4.69
min,43.00,32.70,11.90,23.90,24.30,0.00,3.00,65.00
25%,196.00,47.10,41.70,29.70,27.10,0.00,5.00,77.00
50%,204.00,52.50,47.50,31.00,27.80,0.20,5.00,80.00
75%,210.00,58.30,52.90,32.40,28.30,5.80,6.00,83.00
max,517.00,88.10,67.30,37.50,30.40,216.70,15.00,95.00


In [23]:
print('Distribusi kategori hujan:')
print(df_merged['kategori_hujan'].value_counts().to_string())

Distribusi kategori hujan:
kategori_hujan
Tidak Hujan      226
Sangat Ringan    105
Ringan            68
Sedang            37
Lebat             19
Sangat Lebat       1
Ekstrem            1


## 13. Export ke CSV

In [25]:
OUTPUT = 'dps_bmkg_merged.csv'

df_merged.to_csv(OUTPUT, index=False, date_format='%Y-%m-%d')

print(f'✅ File tersimpan : {OUTPUT}')
print(f'   Baris         : {len(df_merged):,}')
print(f'   Kolom         : {len(df_merged.columns)}')
print(f'   Rentang       : {df_merged["tanggal"].min().date()} s/d {df_merged["tanggal"].max().date()}')
print(f'\nDaftar kolom:')
for i, c in enumerate(df_merged.columns, 1):
    print(f'  {i:2}. {c}')

✅ File tersimpan : dps_bmkg_merged.csv
   Baris         : 457
   Kolom         : 21
   Rentang       : 2025-03-01 s/d 2026-05-31

Daftar kolom:
   1. tanggal
   2. total_flight
   3. early
   4. ontime
   5. delay
   6. cancelled
   7. operated
   8. pct_ontime
   9. pct_delay
  10. avg_delay_menit
  11. suhu_min
  12. suhu_max
  13. suhu_avg
  14. kelembapan
  15. curah_hujan
  16. rr_imputed
  17. kategori_hujan
  18. penyinaran_jam
  19. angin_max_ms
  20. angin_avg_ms
  21. arah_angin


---
## Kamus Kolom `dps_bmkg_merged.csv`

### Data Penerbangan
| Kolom | Keterangan |
|---|---|
| `tanggal` | Tanggal (YYYY-MM-DD) |
| `total_flight` | Total penerbangan departures DPS |
| `early` | Berangkat lebih awal (>1 menit) |
| `ontime` | Tepat waktu (−1 s/d +15 menit) |
| `delay` | Terlambat (>15 menit) |
| `cancelled` | Dibatalkan/diverted |
| `operated` | Total operated (exclude cancelled) |
| `pct_ontime` | % on-time dari operated (%) |
| `pct_delay` | % delay dari operated (%) |
| `avg_delay_menit` | Rata-rata menit keterlambatan |

### Data Cuaca BMKG Ngurah Rai
| Kolom | Keterangan | Satuan |
|---|---|---|
| `suhu_min` | Suhu minimum harian | °C |
| `suhu_max` | Suhu maksimum harian | °C |
| `suhu_avg` | Suhu rata-rata harian | °C |
| `kelembapan` | Kelembapan relatif rata-rata | % |
| `curah_hujan` | Curah hujan harian | mm |
| `rr_imputed` | True jika curah hujan diimputasi | bool |
| `kategori_hujan` | Intensitas hujan (BMKG) | — |
| `penyinaran_jam` | Lama penyinaran matahari | jam |
| `angin_max_ms` | Kecepatan angin maksimum | m/s |
| `angin_avg_ms` | Kecepatan angin rata-rata | m/s |
| `arah_angin` | Arah angin dominan | N/NE/E/… |